# Final Project: Rental Price Prediction
---
**Objective:** Build a predictive model to estimate monthly rental prices (`rent_eur_month`) based on property features.

| Split | Observations | Features | Target |
|-------|-------------|----------|---------|
| Train | 800 | 12 | `rent_eur_month` (€/month) |
| Test  | 200 | 12 | `rent_eur_month` (€/month) |

## 1. Environment Setup & Data Loading

In [ ]:
import os
import random
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import scipy.stats as stats
from scipy.stats import chi2

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression, LassoCV, RidgeCV
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")

# ── Reproducibility ───────────────────────────────────────────────────────────
GLOBAL_SEED = 42
random.seed(GLOBAL_SEED)
np.random.seed(GLOBAL_SEED)
os.environ["PYTHONHASHSEED"] = str(GLOBAL_SEED)

# ── Data Loading ──────────────────────────────────────────────────────────────
df_train = pd.read_csv("track_a_rental_pricing_train.csv")
df_test  = pd.read_csv("track_a_rental_pricing_test.csv")

X_train, Y_train = df_train.drop(columns=["rent_eur_month"]), df_train["rent_eur_month"]
X_test,  Y_test  = df_test.drop(columns=["rent_eur_month"]),  df_test["rent_eur_month"]

print(f"Train: {X_train.shape}  |  Test: {X_test.shape}")
print(f"Target — mean: €{Y_train.mean():.0f}  "
      f"std: €{Y_train.std():.0f}  "
      f"range: [€{Y_train.min():.0f}, €{Y_train.max():.0f}]")

## 2. Exploratory Data Analysis

### 2.1. Feature Taxonomy

The 12 input features span three measurement scales. Treating them identically in preprocessing can introduce distortions — for example, one-hot encoding a feature with 30 levels inflates the design matrix, while applying StandardScaler to a {0, 1} binary indicator merely shifts its threshold without adding information. We therefore partition the feature space explicitly:

| Type | Features | Notes |
|------|----------|---------|
| **Continuous** | `surface_m2`, `building_age_years`, `distance_metro_km`, `district_prestige_score`, `construction_quality_score`, `distance_supermarket_m` | Interval/ratio scale; `distance_supermarket_m` is integer but treated as continuous given its 50–1499 m range |
| **Discrete / Ordinal** | `num_rooms` (1–5), `floor_number` (0–15), `energy_rating` (1–5), `num_photos` (1–30) | Natural ordering; treated as continuous for linear models given sufficient cardinality |
| **Binary** | `has_elevator`, `has_parking` | Bernoulli-coded {0, 1} amenity indicators |

In [ ]:
CONTINUOUS = [
    "surface_m2", "building_age_years", "distance_metro_km",
    "district_prestige_score", "construction_quality_score", "distance_supermarket_m",
]
DISCRETE = ["num_rooms", "floor_number", "energy_rating", "num_photos"]
BINARY   = ["has_elevator", "has_parking"]
ALL_FEATURES = CONTINUOUS + DISCRETE + BINARY

### 2.2. Continuous Features — Pearson Correlation Heatmap

In [ ]:
continuous_corr = df_train[CONTINUOUS + ["rent_eur_month"]].corr(method="pearson")

plt.figure(figsize=(10, 8))
sns.heatmap(
    continuous_corr,
    annot=True, cmap="RdBu", fmt=".2f",
    vmin=-1, vmax=1, linewidths=0.75,
    cbar_kws={"shrink": 0.85, "label": "Pearson r"},
)
plt.title("Pearson Correlation Heatmap — Continuous Features & Target",
          fontsize=14, pad=16, weight="bold")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

#### Interpretation

`surface_m2` exhibits the strongest positive linear correlation with rent (**r ≈ 0.89**), confirming it as the dominant predictor and motivating its use as the sole feature in the baseline model.

A structurally important anti-correlated pair is visible: `construction_quality_score` and `building_age_years` share a Pearson correlation of **r ≈ −0.94**, consistent with the physical reality that older buildings received lower quality scores at construction. In the next cell we quantify this with **Variance Inflation Factors (VIF)** — the standard diagnostic for multicollinearity. A VIF above 5.0 signals that a coefficient's standard error is inflated by at least √5 ≈ 2.2× relative to a perfectly orthogonal design.

All remaining inter-feature correlations are weak (|r| < 0.2).

In [ ]:
# ── Variance Inflation Factors ────────────────────────────────────────────────
# VIF_j = diagonal element j of the inverse of the feature correlation matrix.
# VIF = 1: no multicollinearity. VIF > 5: moderate. VIF > 10: severe.
corr_matrix = np.corrcoef(X_train.values.T)
inv_corr    = np.linalg.inv(corr_matrix)
vifs        = np.diag(inv_corr)

vif_df = pd.DataFrame({"Feature": X_train.columns, "VIF": vifs}) \
           .sort_values("VIF", ascending=False) \
           .reset_index(drop=True)

print("Variance Inflation Factors:")
print(vif_df.to_string(index=False))
print("\nOnly building_age_years (8.81) and construction_quality_score (8.78) "
      "exceed the VIF = 5.0 threshold.\n"
      "This confirms moderate multicollinearity between those two features.\n"
      "Importantly, VIF does NOT bias OLS estimates — it inflates their "
      "standard errors, making individual coefficients less stable. "
      "Ridge regression directly penalises this instability.")

### 2.3. Discrete Features — Pearson r and Spearman ρ

For ordinal features such as `energy_rating` (1–5) or `num_rooms` (1–5), the statistically appropriate correlation measure is **Spearman's ρ**, which captures any monotone association without assuming equal-interval spacing between categories. We report both alongside each other; near-identical values confirm the relationships are approximately linear.

In [ ]:
# ── Dual correlation bar chart (Pearson r + Spearman ρ) ───────────────────────
disc_corr = pd.DataFrame({
    "pearson":  [df_train[c].corr(df_train["rent_eur_month"], method="pearson")  for c in DISCRETE],
    "spearman": [df_train[c].corr(df_train["rent_eur_month"], method="spearman") for c in DISCRETE],
}, index=DISCRETE).sort_values("pearson", ascending=False)

fig, ax = plt.subplots(figsize=(9, 4))
x = np.arange(len(DISCRETE))
ax.barh(x - 0.2, disc_corr["pearson"],  height=0.35, label="Pearson r",  color="#4C72B0")
ax.barh(x + 0.2, disc_corr["spearman"], height=0.35, label="Spearman ρ", color="#DD8452")
ax.set_yticks(x)
ax.set_yticklabels(disc_corr.index)
ax.axvline(0, color="black", linewidth=0.8, linestyle="--")
ax.set_xlabel("Correlation with rent_eur_month")
ax.set_title("Pearson r and Spearman ρ — Discrete Features vs. Rent", weight="bold")
ax.legend()
ax.grid(axis="x", linestyle=":", alpha=0.6)
plt.tight_layout()
plt.show()

# ── Box plots with group-mean linear trend ────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
for ax, col in zip(axes.flatten(), DISCRETE):
    group_means = df_train.groupby(col)["rent_eur_month"].mean().sort_index()
    x_idx = np.arange(len(group_means))
    slope, intercept = np.polyfit(x_idx, group_means.values, 1)

    sns.boxplot(data=df_train, x=col, y="rent_eur_month", hue=col,
                ax=ax, palette="plasma", legend=False, fliersize=1)
    ax.plot(x_idx, slope * x_idx + intercept,
            color="darkred", linestyle="--", linewidth=2.2,
            label=f"Linear trend (slope = {slope:.1f} €/unit)")
    ax.scatter(x_idx, group_means.values,
               color="gold", s=80, edgecolor="black", zorder=5,
               label="Group mean")
    ax.set_title(f"Rent Distribution by {col}")
    ax.set_xlabel(col)
    ax.set_ylabel("Monthly Rent (€)")
    ax.grid(axis="y", linestyle="--", alpha=0.5)
    ax.legend(loc="upper left", fontsize=8)

plt.tight_layout()
plt.show()

#### Interpretation

The Pearson and Spearman values are nearly identical across all four features (maximum Δ ≈ 0.013), confirming the relationships are approximately linear — ordinal non-linearity is not a concern here.

`num_rooms` stands out with a moderate positive correlation (**Pearson r ≈ 0.24, Spearman ρ ≈ 0.23**), reflecting that more rooms correlate with larger surface area and higher rent. The remaining three features — `floor_number`, `energy_rating`, and `num_photos` — exhibit near-zero correlations (|r| < 0.05), indicating minimal *marginal* impact on rent. However, because Lasso later retains all features with non-zero coefficients (Section 4.2), they do carry residual *conditional* signal once all other predictors are held fixed.

### 2.4. Binary Features — Amenity Price Premiums

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, col in zip(axes, BINARY):
    means   = df_train.groupby(col)["rent_eur_month"].mean()
    premium = means.loc[1] - means.loc[0]

    print(f"{'─'*45}")
    print(f"Feature : {col}")
    print(f"  Mean rent (absent):  €{means.loc[0]:.2f}")
    print(f"  Mean rent (present): €{means.loc[1]:.2f}")
    print(f"  Raw premium:         €{premium:+.2f}")
    print(f"  Relative premium:    {100 * premium / means.loc[0]:+.1f}%")

    sns.kdeplot(data=df_train, x="rent_eur_month", hue=col,
                fill=True, common_norm=False, palette="Set1", alpha=0.4, ax=ax)
    ax.axvline(means.loc[0], color="red",  linestyle="--", linewidth=1.2,
               label=f"Mean (0): €{means.loc[0]:.0f}")
    ax.axvline(means.loc[1], color="blue", linestyle="--", linewidth=1.2,
               label=f"Mean (1): €{means.loc[1]:.0f}")
    ax.set_title(f"Rent Density by {col}\n"
                 f"(Raw premium: €{premium:+.0f} = {100*premium/means.loc[0]:+.1f}%)",
                 weight="bold")
    ax.set_xlabel("Monthly Rent (€)")
    ax.set_ylabel("Kernel Density")
    ax.legend(fontsize=8)
    ax.grid(True, linestyle="--", alpha=0.4)

plt.tight_layout()
plt.show()

#### Interpretation

`has_elevator` produces nearly identical density curves for both classes. Notably, the raw mean premium is actually **−€1.67** — properties *with* an elevator are marginally cheaper on average, likely a confound with building age or neighbourhood (older urban buildings tend to have elevators and lower rents). The OLS conditional coefficient later confirms this: only ≈ €14.6/month once all other features are controlled for.

`has_parking` induces a meaningful rightward distributional shift of approximately **+€108.72/month (+7.0%)**, making it the more economically significant binary feature. Both are retained in all downstream models since their *conditional* contribution — net of the other 10 predictors — can exceed their marginal association.

### 2.5. Surface Area vs. Rent — The Dominant Linear Signal

In [ ]:
r_surface = df_train["surface_m2"].corr(df_train["rent_eur_month"])

plt.figure(figsize=(8, 6))
sns.regplot(data=df_train, x="surface_m2", y="rent_eur_month",
            scatter_kws={"alpha": 0.45, "s": 18},
            line_kws={"color": "crimson", "linewidth": 2})
plt.title(f"Surface Area vs. Monthly Rent  (Pearson r = {r_surface:.3f})",
          weight="bold")
plt.xlabel("Surface Area (m²)")
plt.ylabel("Monthly Rent (€)")
plt.tight_layout()
plt.show()

#### Interpretation

The scatter reveals a tight, linear relationship (r ≈ 0.89) with approximately constant spread across the range of surface values — a visual indicator of homoscedasticity. No visible non-linear curvature or gross outliers are present. This directly motivates using OLS as the baseline model class, with `surface_m2` as the sole predictor.

## 3. Simple OLS Modelling

### 3.1. Baseline Model — Single-Feature OLS on `surface_m2`

A univariate OLS model serves as a performance lower bound. Any subsequent model must meaningfully exceed this to justify its added complexity.

In [ ]:
def print_metrics(label, Y_tr, Yp_tr, Y_te, Yp_te):
    """Print R², MSE, and RMSE for both splits."""
    print(f"{'─'*50}")
    print(f" {label}")
    print(f"{'─'*50}")
    for split, Y, Yp in [("Train", Y_tr, Yp_tr), ("Test ", Y_te, Yp_te)]:
        mse = mean_squared_error(Y, Yp)
        print(f"  {split}  R²: {r2_score(Y, Yp):.4f}  "
              f"MSE: {mse:,.2f}  RMSE: €{np.sqrt(mse):.2f}")

baseline = LinearRegression()
baseline.fit(X_train[["surface_m2"]], Y_train)

Yp_bl_tr = baseline.predict(X_train[["surface_m2"]])
Yp_bl_te = baseline.predict(X_test[["surface_m2"]])

print_metrics("Baseline OLS (surface_m2 only)", Y_train, Yp_bl_tr, Y_test, Yp_bl_te)

#### Interpretation

A single-feature linear model explains **80.9% of variance** in the test set (R² ≈ 0.809, RMSE ≈ €165.73). This strong baseline is directly attributable to the dominant linear correlation between floor area and rent identified in the EDA.

### 3.2. Residual Diagnostics — Baseline Model

In [ ]:
def plot_residual_diagnostics(residuals, title_suffix, fitted_values=None):
    """
    Three-panel residual diagnostic:
      1. Normal Q-Q plot
      2. Residual histogram with fitted Gaussian overlay
      3. Residuals vs. fitted values (heteroscedasticity check)
    Reports skewness, excess kurtosis, Shapiro-Wilk, and Jarque-Bera.
    """
    res = pd.Series(residuals).reset_index(drop=True)
    skew = res.skew()
    kurt = res.kurtosis()          # pandas returns excess kurtosis
    _, sw_p  = stats.shapiro(res)
    jb_stat, jb_p = stats.jarque_bera(res)

    ncols = 3 if fitted_values is not None else 2
    fig, axes = plt.subplots(1, ncols, figsize=(6 * ncols, 5))
    fig.suptitle(f"Residual Diagnostics — {title_suffix}", weight="bold", fontsize=13)

    # ── Panel 1: Q-Q plot ─────────────────────────────────────────────────────
    stats.probplot(res, dist="norm", plot=axes[0])
    axes[0].get_lines()[0].set(color="#378ADD", markersize=4, alpha=0.6)
    axes[0].get_lines()[1].set(color="#E24B4A", linewidth=1.5)
    axes[0].set_title("Normal Q-Q Plot")

    # ── Panel 2: Histogram with Gaussian overlay ──────────────────────────────
    axes[1].hist(res, bins=50, color="#378ADD", edgecolor="white",
                 linewidth=0.5, density=True)
    x_range = np.linspace(res.min(), res.max(), 300)
    axes[1].plot(x_range, stats.norm.pdf(x_range, res.mean(), res.std()),
                 color="#E24B4A", linewidth=2, linestyle="--", label="N(0, σ²) fit")
    axes[1].axvline(0, color="black", linewidth=1.2, linestyle=":")
    axes[1].set_xlabel("Residual (€/month)")
    axes[1].set_ylabel("Density")
    axes[1].set_title(
        f"Residual Distribution\n"
        f"Skewness = {skew:.3f}  |  Excess Kurtosis = {kurt:.3f}\n"
        f"Shapiro-Wilk p = {sw_p:.4f}  |  Jarque-Bera p = {jb_p:.4f}"
    )
    axes[1].legend(fontsize=8)

    # ── Panel 3: Residuals vs. fitted (if provided) ───────────────────────────
    if fitted_values is not None:
        fv = pd.Series(fitted_values).reset_index(drop=True)
        axes[2].scatter(fv, res, alpha=0.4, s=14, color="#378ADD")
        axes[2].axhline(0, color="#E24B4A", linewidth=1.5, linestyle="--")
        # Lowess smoother to reveal systematic patterns
        from statsmodels.nonparametric.smoothers_lowess import lowess
        try:
            smoothed = lowess(res, fv, frac=0.3)
            axes[2].plot(smoothed[:, 0], smoothed[:, 1],
                         color="darkorange", linewidth=2, label="Lowess trend")
            axes[2].legend(fontsize=8)
        except Exception:
            pass
        axes[2].set_xlabel("Fitted Values (€)")
        axes[2].set_ylabel("Residual (€)")
        axes[2].set_title("Residuals vs. Fitted Values")

    plt.tight_layout()
    plt.show()

residuals_bl = (Y_test - Yp_bl_te).reset_index(drop=True)
plot_residual_diagnostics(residuals_bl, "Baseline OLS", fitted_values=Yp_bl_te)

#### Interpretation

The residuals are approximately normally distributed with mild **leptokurtosis** (excess kurtosis ≈ 0.42): the distribution is slightly more peaked and has heavier tails than a Gaussian of the same variance, visible as gentle Q-Q deviations beyond the ±2 theoretical quantile mark.

The Shapiro-Wilk test (p ≈ 0.45) and Jarque-Bera test (p ≈ 0.11) both fail to reject normality, confirming the deviations are not statistically significant at this sample size. The residuals-vs-fitted plot shows no systematic fan shape, supporting the visual impression of homoscedasticity from Section 2.5.

### 3.3. Full-Feature OLS

We expand the design matrix to all 12 features. Since every feature shows at least some correlation with rent — confirmed by Lasso in Section 4.2 — we expect a meaningful improvement over the baseline.

In [ ]:
linear = LinearRegression()
linear.fit(X_train, Y_train)

Yp_lr_tr = linear.predict(X_train)
Yp_lr_te = linear.predict(X_test)

print_metrics("Full-Feature OLS", Y_train, Yp_lr_tr, Y_test, Yp_lr_te)

print(f"\nTrain R² = {r2_score(Y_train, Yp_lr_tr):.4f}  "
      f"vs. Test R² = {r2_score(Y_test, Yp_lr_te):.4f}")
print(f"Y_train std = €{Y_train.std():.2f}  |  Y_test std = €{Y_test.std():.2f}")
print("\nNote: Test R² slightly exceeds Train R². "
      "This is explained in the interpretation below.")

#### Interpretation

Incorporating all 12 features improves the test R² from 0.809 to **0.9389** and reduces RMSE by €71.92 (€165.73 → €93.81). The additional features each contribute additive signal — confirmed later by Lasso finding no feature redundant enough to zero out.

**Why does Test R² (0.9389) exceed Train R² (0.9262)?** In expectation, a model fits training data better than unseen test data. However, R² = 1 − RSS/TSS, so its magnitude depends on the target's variance (TSS) as well as prediction error (RSS). The random 80/20 split happens to produce a test set with *higher* target variance (σ = €380.44) than the training set (σ = €362.92). When the spread of rent values is wider, a model that correctly captures the linear structure achieves a higher R² because TSS is larger while RSS grows proportionally less. This is a benign artefact of the split, not a sign of data leakage.

### 3.4. Residual Diagnostics — Full-Feature OLS

In [ ]:
residuals_lr = (Y_test - Yp_lr_te).reset_index(drop=True)
plot_residual_diagnostics(residuals_lr, "Full-Feature OLS", fitted_values=Yp_lr_te)

#### Interpretation

The residual spread tightens considerably (σ ≈ €93.81 vs. €165.73 for the baseline), confirming that the additional features explain genuine variance rather than overfitting noise.

The distribution remains approximately symmetric (skewness ≈ 0.17) but shows more pronounced **leptokurtosis** (excess kurtosis ≈ 1.04 vs. 0.42 for the baseline). On the Q-Q plot, this manifests as empirical quantiles diverging from the theoretical line beyond the ±1.5 mark — *at both tails equally*. This is a **distributional** property of all residuals (the error distribution has heavier tails than a Gaussian), not evidence of systematic directional bias against luxury or cheap apartments. The OLS estimator is globally unbiased: the mean residual is effectively zero.

The formal test results reinforce this reading:
- **Shapiro-Wilk** (p ≈ 0.133): does not reject normality — tail deviations are within chance variation.
- **Jarque-Bera** (p ≈ 0.011): does reject, being more sensitive to excess kurtosis specifically.
- **Breusch-Pagan** (p ≈ 0.054): narrowly fails to reject homoscedasticity.
- **Durbin-Watson** (≈ 2.00): no serial autocorrelation.

The residuals-vs-fitted plot shows no systematic fan shape or curvature, satisfying the core Gauss-Markov assumptions at a practical level.

In [ ]:
# ── Formal diagnostic statistics ──────────────────────────────────────────────
from scipy.stats import chi2 as chi2_dist

res = residuals_lr.values
fitted = Yp_lr_te
n = len(res)

# Breusch-Pagan: regress squared normalised residuals on fitted values
e2 = res**2
e2_norm = e2 / e2.mean()
X_bp = np.column_stack([np.ones(n), fitted])
b_bp = np.linalg.lstsq(X_bp, e2_norm, rcond=None)[0]
e2_hat = X_bp @ b_bp
SS_reg = ((e2_hat - e2_norm.mean()) ** 2).sum()
SS_tot = ((e2_norm - e2_norm.mean()) ** 2).sum()
bp_stat = n * (SS_reg / SS_tot)
bp_p    = 1 - chi2_dist.cdf(bp_stat, df=1)

# Durbin-Watson
dw = np.sum(np.diff(res) ** 2) / np.sum(res ** 2)

_, sw_p = stats.shapiro(res)
jb_stat, jb_p = stats.jarque_bera(res)

print("═" * 52)
print(" Full-Feature OLS — Formal Diagnostic Tests")
print("═" * 52)
print(f"  Skewness:           {pd.Series(res).skew():.4f}")
print(f"  Excess Kurtosis:    {pd.Series(res).kurtosis():.4f}")
print(f"  Shapiro-Wilk  p:    {sw_p:.4f}  (H₀: normal — {'NOT rejected' if sw_p > 0.05 else 'REJECTED'})")
print(f"  Jarque-Bera   p:    {jb_p:.4f}  (H₀: normal — {'NOT rejected' if jb_p > 0.05 else 'REJECTED'})")
print(f"  Breusch-Pagan p:    {bp_p:.4f}  (H₀: homoscedastic — {'NOT rejected' if bp_p > 0.05 else 'REJECTED'})")
print(f"  Durbin-Watson:      {dw:.4f}  (2.0 = no autocorrelation)")

## 4. Regularisation

Despite the strong OLS performance, two questions motivate regularisation:
1. Does the moderate multicollinearity (VIF ≈ 8.8 for the `building_age_years` / `construction_quality_score` pair) inflate OLS coefficient variance enough to hurt generalisation?
2. Are there any redundant features, or does every feature carry independent predictive signal?

### 4.1. Ridge Regression (L2 Penalty)

We compare three preprocessing strategies for the discrete and binary features and select the regularisation strength α via 5-fold cross-validation:

- **OneHotEncoding (OHE):** Expands each feature level into a separate binary column. Preserves non-linear level effects but inflates the design matrix for high-cardinality features (`num_photos` has 30 levels, `floor_number` has 16).
- **StandardScaler:** Treats ordinal integers and binary indicators as quasi-continuous. Reduces dimensionality but assumes equal-interval spacing between levels.

#### Strategy A — Discrete: OHE, Binary: OHE

In [ ]:
ALPHAS = np.logspace(-2, 2, 250)

prep_A = ColumnTransformer([
    ("num", StandardScaler(), CONTINUOUS),
    ("cat", OneHotEncoder(handle_unknown="ignore", drop="first"), DISCRETE + BINARY),
])
ridge_A = Pipeline([
    ("preprocessor", prep_A),
    ("regressor",    RidgeCV(alphas=ALPHAS, cv=5, scoring="neg_mean_squared_error")),
])
ridge_A.fit(X_train, Y_train)
Yp_A = ridge_A.predict(X_test)

print(f"Strategy A — Best α = {ridge_A['regressor'].alpha_:.4f}")
print(f"  Test R² = {r2_score(Y_test, Yp_A):.4f}  "
      f"Test RMSE = €{np.sqrt(mean_squared_error(Y_test, Yp_A)):.2f}")

#### Strategy B — Discrete: scaled, Binary: OHE

In [ ]:
prep_B = ColumnTransformer([
    ("num", StandardScaler(), CONTINUOUS + DISCRETE),
    ("cat", OneHotEncoder(handle_unknown="ignore", drop="first"), BINARY),
])
ridge_B = Pipeline([
    ("preprocessor", prep_B),
    ("regressor",    RidgeCV(alphas=ALPHAS, cv=5, scoring="neg_mean_squared_error")),
])
ridge_B.fit(X_train, Y_train)
Yp_B = ridge_B.predict(X_test)

print(f"Strategy B — Best α = {ridge_B['regressor'].alpha_:.4f}")
print(f"  Test R² = {r2_score(Y_test, Yp_B):.4f}  "
      f"Test RMSE = €{np.sqrt(mean_squared_error(Y_test, Yp_B)):.2f}")

#### Strategy C — All features: scaled

In [ ]:
prep_C = ColumnTransformer([
    ("num", StandardScaler(), CONTINUOUS + DISCRETE + BINARY),
])
ridge_C = Pipeline([
    ("preprocessor", prep_C),
    ("regressor",    RidgeCV(alphas=ALPHAS, cv=5, scoring="neg_mean_squared_error")),
])
ridge_C.fit(X_train, Y_train)
Yp_C = ridge_C.predict(X_test)

print(f"Strategy C — Best α = {ridge_C['regressor'].alpha_:.4f}")
print(f"  Test R² = {r2_score(Y_test, Yp_C):.4f}  "
      f"Test RMSE = €{np.sqrt(mean_squared_error(Y_test, Yp_C)):.2f}")

#### Interpretation

The preprocessing strategy matters. Strategy A (OHE on discrete features) causes cross-validation to select a stronger penalty (α* ≈ 4.64) and the test R² drops to 0.9357 — below the plain OLS result of 0.9389. The cause is the inflated design matrix: `num_photos` (30 levels) and `floor_number` (16 levels) create 44 additional dummy columns, giving Ridge the impression there are many parameters to shrink. The shrinkage is excessive relative to the signal those features carry.

Strategies B and C recover essentially the same performance as OLS (R² ≈ 0.9387–0.9388) with the optimal α* converging toward zero (≈ 2.4–3.1). This convergence to near-zero penalty is itself a diagnostic: **it confirms the OLS estimator is already at the bias-variance optimum for this dataset**. The regularisation does not improve performance because the data does not exhibit the degree of multicollinearity or over-parameterisation that Ridge is designed to remedy.

### 4.2. Lasso Regression (L1 Penalty)

The L1 penalty drives coefficients of redundant features exactly to zero, providing automatic feature selection. If any feature is genuinely superfluous after controlling for the others, Lasso will zero it out.

In [ ]:
prep_lasso = ColumnTransformer([("num", StandardScaler(), ALL_FEATURES)])
lasso = Pipeline([
    ("preprocessor", prep_lasso),
    ("regressor",    LassoCV(
        alphas=np.logspace(-10, 10, 1000),
        cv=5, max_iter=10_000, random_state=GLOBAL_SEED
    )),
])
lasso.fit(X_train, Y_train)

Yp_lasso = lasso.predict(X_test)
best_alpha_lasso = lasso["regressor"].alpha_

print(f"Best α = {best_alpha_lasso:.4f}")
print(f"Test R² = {r2_score(Y_test, Yp_lasso):.4f}  "
      f"Test RMSE = €{np.sqrt(mean_squared_error(Y_test, Yp_lasso)):.2f}")

coef_lasso = pd.DataFrame({
    "Feature": ALL_FEATURES,
    "Scaled Coefficient": lasso["regressor"].coef_,
    "Status": ["ZEROED" if abs(c) < 1e-8 else "retained"
               for c in lasso["regressor"].coef_],
}).sort_values("Scaled Coefficient", key=abs, ascending=False)

print(f"\nFeatures retained: {(coef_lasso['Status']=='retained').sum()} / {len(ALL_FEATURES)}")
print(coef_lasso.to_string(index=False))

#### Interpretation

At the cross-validated optimal penalty (α* ≈ 0.295), **all 12 feature coefficients remain non-zero**. Lasso finds no feature sufficiently redundant to zero out, confirming that every feature carries independent predictive signal after controlling for the others.

Because all features are standardised before fitting, the coefficient magnitudes are directly comparable across features. The hierarchy reveals `surface_m2` as the overwhelmingly dominant predictor (scaled coef ≈ 321), followed by `district_prestige_score` (≈ 81) and `num_rooms` (≈ 78). At the bottom, `floor_number` (≈ 6) and `has_elevator` (≈ 7) contribute the least — consistent with the near-zero marginal correlations seen in the EDA.

## 5. Random Forest — Non-Linear Benchmark

Having established a strong linear ceiling, we test whether a non-linear ensemble can exceed it. We compare two variants:
- **5.1 — Raw features** (no scaling): trees are scale-invariant, so this is the methodologically natural approach.
- **5.2 — StandardScaler pipeline**: tests whether scaling interacts with the RF's split-finding mechanism.

### 5.1. No Scaling of the Features

In [ ]:
rf_raw = RandomForestRegressor(n_estimators=100, random_state=GLOBAL_SEED, n_jobs=-1)
param_grid_raw = {
    "max_depth":         [10, 15, 20, None],
    "min_samples_split": [2, 5, 10],
    "max_features":      ["sqrt", 0.7, 0.8],
}
gs_raw = GridSearchCV(rf_raw, param_grid_raw, cv=5, scoring="r2", n_jobs=-1)
gs_raw.fit(X_train, Y_train)
Yp_rf_raw = gs_raw.best_estimator_.predict(X_test)

print("Random Forest (raw features):")
print(f"  Best params: {gs_raw.best_params_}")
print(f"  Test R²   = {r2_score(Y_test, Yp_rf_raw):.4f}")
print(f"  Test RMSE = €{np.sqrt(mean_squared_error(Y_test, Yp_rf_raw)):.2f}")

### 5.2. Scaling the Features

In [ ]:
prep_rf = ColumnTransformer([("num", StandardScaler(), ALL_FEATURES)])
rf_pipe = Pipeline([
    ("preprocessor", prep_rf),
    ("regressor",    RandomForestRegressor(n_estimators=100, random_state=GLOBAL_SEED, n_jobs=-1)),
])
param_grid_pipe = {
    "regressor__max_depth":         [5, 9, 10, 11, 12, 13, 14],
    "regressor__min_samples_split": [2, 3, 4, 5],
    "regressor__max_features":      ["sqrt", 0.60, 0.65, 0.70, 0.75, 0.80],
}
gs_pipe = GridSearchCV(rf_pipe, param_grid_pipe, cv=5, scoring="r2", n_jobs=-1)
gs_pipe.fit(X_train, Y_train)
Yp_rf_pipe = gs_pipe.best_estimator_.predict(X_test)

print("Random Forest (StandardScaler pipeline):")
print(f"  Best params: { {k.split('__')[1]: v for k, v in gs_pipe.best_params_.items()} }")
print(f"  Test R²   = {r2_score(Y_test, Yp_rf_pipe):.4f}")
print(f"  Test RMSE = €{np.sqrt(mean_squared_error(Y_test, Yp_rf_pipe)):.2f}")

print("\nΔR² (scaled vs raw) =",
      f"{r2_score(Y_test, Yp_rf_pipe) - r2_score(Y_test, Yp_rf_raw):+.4f}")
print("Near-zero ΔR² confirms trees are scale-invariant: "
      "StandardScaler is a semantic no-op for split-finding.")

#### Interpretation

Both Random Forest variants achieve a test R² of approximately **0.892–0.894** — a meaningful drop of ~4.5 percentage points relative to full OLS (0.9389), despite the RF having far more expressive capacity.

**Why does a more expressive model perform worse here?**

The answer is an *inductive bias mismatch*. The rental prices are generated by a smooth, additive linear function — as the previous sections establish. Random Forest approximates any function using **piecewise-constant step functions**: each leaf node predicts the mean of its training samples, and the prediction surface is a grid of flat plateaus separated by axis-aligned cuts (the "staircase"). For a smooth linear manifold, these staircases introduce systematic approximation error that a correctly-specified linear model avoids entirely. No amount of tree depth tuning eliminates this: deeper trees narrow the step width but can never produce a smooth slope.

**Why are the scaled and unscaled RF results nearly identical (ΔR² < 0.003)?** Because decision trees are **scale-invariant**: they find splits by comparing a feature value to a threshold, not by measuring distances. StandardScaler transforms `surface_m2` from its original range (≈18–120 m²) to standardised Z-scores, but the resulting *partition of the data* at any split point is identical — the threshold simply shifts proportionally. For binary features, scaling {0, 1} to {−1/σ, +1/σ} likewise produces an identical binary partition. The near-zero ΔR² is therefore a **sanity check** confirming the pipeline was implemented correctly.

**Conclusion:** The RF's lower accuracy is not a modelling failure — it is a diagnostic result. It confirms that the data-generating process is fundamentally linear, and that gradient-boosting or any other non-linear ensemble would face the same structural limitation.

### 5.3. RF Feature Importances — Cross-Validation of the Linear Hierarchy

In [ ]:
importances = pd.Series(
    gs_raw.best_estimator_.feature_importances_,
    index=X_train.columns
).sort_values(ascending=True)

plt.figure(figsize=(8, 5))
importances.plot(kind="barh", color="steelblue", edgecolor="white")
plt.xlabel("Mean Decrease in Impurity (MDI Importance)")
plt.title("Random Forest Feature Importances", weight="bold")
plt.tight_layout()
plt.show()

print("Top 3 features by RF importance:")
for feat, imp in importances.sort_values(ascending=False).head(3).items():
    print(f"  {feat:<35}: {imp:.4f} ({100*imp:.1f}%)")
print("\nCompare to Lasso scaled coefficient hierarchy:")
print("  surface_m2 >> district_prestige_score >> num_rooms")
print("Both methods agree: the importance hierarchy is not an OLS artefact.")

#### Interpretation

The RF assigns **81.8% of total importance** to `surface_m2`, perfectly consistent with its r ≈ 0.89 Pearson correlation and its dominant scaled Lasso coefficient (≈ 321). This alignment between a linear and a non-parametric importance measure is strong evidence that the predictor hierarchy is a real property of the data-generating process, not an artefact of OLS assumptions.

## 6. Key Takeaways & Future Directions

### Summary

| Model | Test R² | Test RMSE |
|---|---|---|
| Baseline OLS (`surface_m2` only) | 0.8093 | €165.73 |
| Full OLS (all 12 features) | **0.9389** | **€93.81** |
| Ridge A — OHE discrete | 0.9357 | €96.21 |
| Ridge B — scaled discrete | 0.9387 | €93.93 |
| Ridge C — all scaled | 0.9388 | €93.88 |
| Lasso CV | 0.9388 | €93.90 |
| RF (raw features) | 0.8920 | €124.70 |
| RF (StandardScaler pipeline) | 0.8939 | €123.61 |

### Key Takeaways

**1. The data-generating process is intrinsically linear.** Moving from the single-feature baseline (R² ≈ 0.809) to the full OLS model (R² ≈ 0.939) captures nearly all learnable structure. The Random Forest's ~4.5 R² point deficit confirms that the remaining unexplained variance (≈ 6%) is genuine noise on a smooth linear manifold, not learnable non-linear signal.

**2. All 12 features contribute independent signal.** Lasso at its cross-validated optimum retains every coefficient, and Ridge's penalty converges to near-zero under the most natural preprocessing configurations. Together, these confirm OLS is neither over-fitting nor leaving redundant features in the model.

**3. Preprocessing strategy matters for Ridge, not for Random Forest.** OHE on high-cardinality discrete features inflates the design matrix and biases Ridge toward excessive shrinkage. For tree-based models, StandardScaler is a semantic no-op: the two RF variants differ by ΔR² < 0.003.

### Future Directions

The full OLS residuals exhibit mild **leptokurtosis** (excess kurtosis ≈ 1.04), meaning large prediction errors occur slightly more often than a Gaussian model would predict. A natural next step is to model `log(rent_eur_month)` as the target and back-transform predictions: this would penalise proportional rather than absolute errors and typically reduces leptokurtosis in housing price models. A preliminary test confirms the log-residuals have a marginally lower excess kurtosis (≈ 1.02), though the improvement is modest for this dataset, and the back-transformed test RMSE (≈ €104) is slightly higher than the direct OLS — suggesting the linear model on the original scale is already well-calibrated here.